# **Tokenisation & Encodage des données**

## **Imports**

In [13]:
# Imports
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import random
import numpy as np

import sys
import os
import importlib

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [14]:
# Définir la racine du projet
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [16]:
# Custom imports
from utils.dataset import MedQADataset

## **Chargement des `splits`**

In [2]:
# Charger les splits
train_df = pd.read_csv('../data/train.csv')
val_df = pd.read_csv('../data/val.csv')
test_df = pd.read_csv('../data/test.csv')

print(f"Train: {len(train_df)}")
print(f"Val: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 11494
Val: 2431
Test: 2434


## **Configurations pour `Mistral-7B` + `QLoRA`**

In [4]:
# Configuration
MODEL_NAME = "mistralai/Mistral-7B-v0.1" 
MAX_LENGTH = 512  # Longueur maximale des séquences
BATCH_SIZE = 4  # À ajuster selon mémoire GPU

In [7]:
# Authentification HuggingFace
from huggingface_hub import login
login()

### **1. Chargement du `Tokenizer`**

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # Configuration du token de padding

print("Tokenizer chargé avec succès")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

Tokenizer chargé avec succès
Vocab size: 32000
Pad token: </s>
EOS token: </s>


In [9]:
# Tester le format d'instruction Mistral
example_question = train_df['question'].iloc[0]
example_answer = train_df['answer'].iloc[0]

# Format Mistral: <s>[INST] Question [/INST] Réponse </s>
formatted_text = f"<s>[INST] {example_question} [/INST] {example_answer} </s>"

tokens = tokenizer(formatted_text, return_tensors="pt")
print(f"Question: {example_question[:50]}...")
print(f"Réponse: {example_answer[:50]}...")
print(f"Texte formaté (début): {formatted_text[:100]}...")
print(f"Longueur en tokens: {tokens['input_ids'].shape[1]}")
print(f"Tokens: {tokens['input_ids'][0][:10]}...")  # Premiers tokens

Question: What causes Glaucoma ?...
Réponse: Nearly 2.7 million people have glaucoma, a leading...
Texte formaté (début): <s>[INST] What causes Glaucoma ? [/INST] Nearly 2.7 million people have glaucoma, a leading cause of...
Longueur en tokens: 306
Tokens: tensor([    1,     1, 28792, 16289, 28793,  1824, 10110,  3651,   581,   675])...


### **2. Création du `dataset` personnalisé**

In [17]:
# Créer une instance du dataset
train_dataset = MedQADataset(train_df, tokenizer, max_length=512)

# Tester un échantillon
sample = train_dataset[0]
print("Input IDs shape:", sample['input_ids'].shape)
print("Attention mask shape:", sample['attention_mask'].shape)
print("Labels shape:", sample['labels'].shape)
print("\nNombre de tokens masqués (question):", (sample['labels'] == -100).sum().item())
print("Nombre de tokens non masqués (réponse):", (sample['labels'] != -100).sum().item())


Input IDs shape: torch.Size([512])
Attention mask shape: torch.Size([512])
Labels shape: torch.Size([512])

Nombre de tokens masqués (question): 15
Nombre de tokens non masqués (réponse): 497


### **3. Création des `DataLoaders`**

In [18]:
# Création des datasets
train_dataset = MedQADataset(train_df, tokenizer, max_length=512)
val_dataset = MedQADataset(val_df, tokenizer, max_length=512)
test_dataset = MedQADataset(test_df, tokenizer, max_length=512)

# DataLoaders
BATCH_SIZE = 4  # Ajuster selon mémoire GPU

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 2874
Validation batches: 608
Test batches: 609


### **4. Test d'un batch**

In [19]:
# Tester un batch
batch = next(iter(train_loader))
print("Batch keys:", batch.keys())
print(f"input_ids shape: {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"labels shape: {batch['labels'].shape}")
print(f"\nExemple de labels (premiers tokens):")
print(batch['labels'][0][:20])

Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([4, 512])
attention_mask shape: torch.Size([4, 512])
labels shape: torch.Size([4, 512])

Exemple de labels (premiers tokens):
tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100])


**Les `-100` confirment que la partie question est bien masquée (ignorée dans la loss).**

## **Sauvegarde des `datasets tokenizés`**

In [20]:
import pickle

# Sauvegarde des datasets
with open('../data/train_dataset.pkl', 'wb') as f:
    pickle.dump(train_dataset, f)
with open('../data/val_dataset.pkl', 'wb') as f:
    pickle.dump(val_dataset, f)
with open('../data/test_dataset.pkl', 'wb') as f:
    pickle.dump(test_dataset, f)

print("Datasets sauvegardés")

Datasets sauvegardés


## **Résumé `tokenization`**

1. Le dataset `MedQuAD` nettoyé (16359 exemples) a été tokenizé avec le tokenizer `Mistral-7B-v0.1`. Les séquences sont paddées/truncatées à 512 tokens avec un batch size de 4.

2. Les 14979 questions uniques réparties en train (70%), validation (15%) et test (15%) ont été converties au format `<s>[INST] question [/INST] réponse </s>`. La partie question est masquée (-100) dans les labels pour que la loss ne soit calculée que sur les réponses.

3. Trois DataLoaders sont prêts : train (2874 batches), validation (608 batches) et test (609 batches). Les datasets ont été sauvegardés pour une utilisation ultérieure.